# Wearable physiology during an influenza challenge

Compare wearable heart rate (HR) and sleep-derived physiology in the
[CIVICs 20-0005](https://clinicaltrials.gov/study/NCT04978454) and
[Exhale](https://clinicaltrials.gov/study/NCT06476275) influenza challenge studies.
The workflow combines participant metadata with wearable measurements, checks
wear time, normalizes HR, and compares trajectories across clinical cohorts.

Run cells from top to bottom from the repository root. Input paths are configured
in the first cell. Sleep records are read from a separate Oura data directory.
Required packages: NumPy, pandas, Matplotlib, SciPy, Pingouin, and PyArrow.

Relative day 0 is the inoculation date; elapsed hour 0 is the inoculation timestamp.
Individual sections use different baseline and nighttime definitions, documented
below. Figure sample sizes are fixed labels from the existing analysis and are
not recalculated after filtering.

1. Check wear time and apply fixed participant exclusions.
2. Define clinical cohorts and summarize demographics.
3. Normalize HR against each participant's pre-inoculation baseline.
4. Compare repeated daytime and nighttime measurements in the SI cohort.
5. Summarize raw daytime and nighttime HR across cohorts.
6. Visualize clock-time profiles and continuous HR trajectories.
7. Align nocturnal HR with viral load and symptom scores.
8. Extract HR within recorded sleep windows and summarize sleep physiology.

This notebook has no saved results; numerical findings require execution with
all input files available.

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg
from matplotlib.lines import Line2D
from scipy.ndimage import gaussian_filter1d

import warnings
warnings.filterwarnings("ignore")

# Paths are relative to the repository root, except the external Oura archive.
DATA_DIR = Path("data")
OURA_ROOT = Path("/media/nvme1/IW-modeling")
CIVICS_SLEEP_DIR = OURA_ROOT / "CIVICs_20_0005" / "Oura"
EXHALE_SLEEP_DIR = OURA_ROOT / "Exhale" / "Oura"

test_info = pd.read_csv("test_information.csv")

## 0. Load wearable heart rate

Load the two study tables at their original sampling intervals: 15 seconds for
CIVICs and one minute for Exhale. The analysis expects `Participant ID`, `dateTime`
(as a datetime column), `relative_day_to_inoculation`, and `hr`.

In [ ]:
CIVICs_HR_full_df = pd.read_parquet(DATA_DIR / "CIVICs_HR_full_df.parquet")
Exhale_HR_full_df = pd.read_parquet(DATA_DIR / "Exhale_HR_full_df.parquet")

## 1. Check wear time and apply fixed exclusions

Estimate coverage on observed days -2 through 7 by dividing the number of complete
rows by the expected sample count. Rows with any missing value are excluded for
this check. Full-day denominators are 5,760 (15 seconds) and 1,440 (one minute);
the inclusive 00:00-08:00 window uses 1,921 and 481, respectively.

A day passes with coverage strictly greater than 0.7, and a nighttime window
passes with coverage strictly greater than 0.5. The diagnostic checks baseline
`day < 1` and post-inoculation `day >= 0`, so day 0 is present in both windows.
It reports baseline insufficiency only when both baseline counts are below two.
The post-inoculation check runs only when at least two baseline full days pass;
it reports insufficiency when fewer than four full days and five nights pass.
Entirely absent days are not added to the coverage table.

These diagnostics print messages; the explicit exclusion list below determines
which participants are removed from subsequent HR analyses.

In [ ]:
def calculate_weartime(
    heart_rate_df: pd.DataFrame, sampling_frequency: str
) -> pd.DataFrame:
    """Return daily and 00:00-08:00 coverage for observed days -2 through 7.

    Count complete rows, preserving the original missing-data rule and inclusive
    nighttime endpoints. Coverage assumes one row per scheduled measurement.
    """
    expected_samples = {"15s": (5760, 1921), "1min": (1440, 481)}
    if sampling_frequency not in expected_samples:
        raise ValueError("sampling_frequency must be '15s' or '1min'.")
    daily_samples, nightly_samples = expected_samples[sampling_frequency]
    night_start = pd.Timestamp("00:00:00").time()
    night_end = pd.Timestamp("08:00:00").time()
    records = []

    for participant in heart_rate_df["Participant ID"].unique():
        participant_df = heart_rate_df.loc[
            heart_rate_df["Participant ID"] == participant
        ].dropna().copy()
        participant_df["time"] = participant_df["dateTime"].dt.time
        for day in participant_df["relative_day_to_inoculation"].unique():
            if not -2 <= day <= 7:
                continue
            day_df = participant_df.loc[
                participant_df["relative_day_to_inoculation"] == day
            ]
            night_df = day_df.loc[day_df["time"].between(night_start, night_end)]
            records.append({
                "participant_id": participant,
                "relative_day_to_inoculation": day,
                "weartime": len(day_df) / daily_samples,
                "sleep_weartime": len(night_df) / nightly_samples,
            })
    return pd.DataFrame(records, columns=[
        "participant_id", "relative_day_to_inoculation", "weartime", "sleep_weartime"
    ])


def weartime_QC(
    weartime_df: pd.DataFrame,
    weartime_threshold: float = 0.7,
    sleep_weartime_threshold: float = 0.5,
) -> None:
    """Print coverage diagnostics using the original nested decision rules."""
    # Work on a copy so inspecting QC does not mutate the caller's table.
    coverage = weartime_df.copy()
    coverage["weartime_flag"] = coverage["weartime"] > weartime_threshold
    coverage["sleep_weartime_flag"] = (
        coverage["sleep_weartime"] > sleep_weartime_threshold
    )
    for participant, participant_df in coverage.groupby("participant_id", sort=False):
        baseline = participant_df.loc[participant_df["relative_day_to_inoculation"] < 1]
        post = participant_df.loc[participant_df["relative_day_to_inoculation"] >= 0]
        if baseline["weartime_flag"].sum() < 2:
            if baseline["sleep_weartime_flag"].sum() < 2:
                print(f"Patient {participant} has insufficient pre-inoculation wear time and sleep data.")
        elif post["weartime_flag"].sum() < 4:
            if post["sleep_weartime_flag"].sum() < 5:
                print(f"Patient {participant} has insufficient post-inoculation sleep data.")

In [ ]:
weartime_QC(calculate_weartime(CIVICs_HR_full_df, sampling_frequency='15s'))
weartime_QC(calculate_weartime(Exhale_HR_full_df, sampling_frequency='1min'))

### 1.1 Apply participant exclusions and combine studies

Remove the fixed list of participants identified by wear-time review or study
withdrawal, then concatenate both study tables. Keep HR observations from day -31
onward so earlier pre-inoculation data can contribute to baseline normalization.
The cohort definitions in the next section still use the full metadata table.

In [ ]:
CIVICS_EXCLUDED_IDS = {
    "C02DU039", "C02DU010", "C02UM043", "C02UM055",
    "C02DU042", "C02DU003", "C01UM026", "C02DU030",
}
EXHALE_EXCLUDED_IDS = {"EXC1DU001", "EXC1DU008"}

CIVICs_HR_full_df = CIVICs_HR_full_df.loc[
    ~CIVICs_HR_full_df["Participant ID"].isin(CIVICS_EXCLUDED_IDS)
].reset_index(drop=True)
Exhale_HR_full_df = Exhale_HR_full_df.loc[
    ~Exhale_HR_full_df["Participant ID"].isin(EXHALE_EXCLUDED_IDS)
].reset_index(drop=True)

In [ ]:
# merge the two HR dataframes together after QC
final_hr_df = pd.concat([CIVICs_HR_full_df, Exhale_HR_full_df], ignore_index=True)
final_hr_df = final_hr_df[final_hr_df['relative_day_to_inoculation'] >= -31].reset_index(drop=True)

## 2. Define cohorts and summarize demographics

Define inoculum groups (`FLUH`, `FLUM`, `FLUL`, and `SHAM`) and four
infection/symptom groups: symptomatic infected (SI), asymptomatic infected (AI),
symptomatic uninfected (SU), and asymptomatic uninfected (AU).
The combined uninfected cohort explicitly excludes sham participants; the four
`INFSYM` groups follow the labels in `test_info` without that extra exclusion.

In [ ]:
FLUH = test_info[test_info['Dosage Name'] == 'FLUH']
FLUM = test_info[test_info['Dosage Name'] == 'FLUM']
FLUL = test_info[test_info['Dosage Name'] == 'FLUL']
SHAM = test_info[test_info['Dosage Name'] == 'SHAM']

SI = test_info[test_info['INFSYM'] == 'Symptomatic and Infected'].reset_index(drop=True)
AI = test_info[test_info['INFSYM'] == 'Asymptomatic and Infected'].reset_index(drop=True)
SU = test_info[test_info['INFSYM'] == 'Symptomatic and Uninfected'].reset_index(drop=True)
AU = test_info[test_info['INFSYM'] == 'Asymptomatic and Uninfected'].reset_index(drop=True)
shams_id = test_info[test_info['Dosage Name'] == 'SHAM']['Participant ID'].tolist()

infected = test_info[test_info['INFSTAT'] == 'Infected'].reset_index(drop=True)
uninfected = test_info[(test_info['INFSTAT'] == 'Uninfected') & (test_info['Dosage Name'] != 'SHAM')].reset_index(drop=True)

### 2.1 Demographic summary helper

`show_demographics("All")` or a cohort code prints sex, race, ethnicity, infection,
symptom, and vaccine counts, plus age and BMI as mean (SD) and median (Q1-Q3).
These summaries describe metadata cohorts before wearable exclusions. The cell
below defines the helper; call it for the cohorts you want to inspect.

In [ ]:
def show_demographics(group: str) -> None:
    '''
    Show demographics for a specific group of participants.
    '''
    cohorts = {
        "FLUH": FLUH, "FLUM": FLUM, "FLUL": FLUL, "SHAM": SHAM,
        "SI": SI, "AI": AI, "SU": SU, "AU": AU, "All": test_info,
    }
    if group not in cohorts:
        raise ValueError(f"Unknown cohort {group!r}; choose from {list(cohorts)}.")
    df = cohorts[group]
    print(f"Demographics for group {group}:")

    # Count number of males and females
    gender_counts = df['SEX'].value_counts()

    # Count race and ethnicity
    race_counts = df['RACE'].str.lower().value_counts()
    ethnicity_counts = df['ETHNIC'].str.lower().value_counts()


    # Calculate average age and BMI with std for the entire group
    age_mean = df['AGE'].mean()
    age_median = df['AGE'].median()
    age_std = df['AGE'].std()
    age_iqr_l = df['AGE'].quantile(0.25)
    age_iqr_u = df['AGE'].quantile(0.75)
    bmi_mean = df['BMI'].mean()
    bmi_std = df['BMI'].std()
    bmi_median = df['BMI'].median()
    bmi_iqr_l = df['BMI'].quantile(0.25)
    bmi_iqr_u = df['BMI'].quantile(0.75)

    print("Number of participants by gender:")
    print(gender_counts)
    print("\nOverall average age: {:.1f} ({:.1f})".format(age_mean, age_std))
    print("Overall median age: {:.1f} ({:.1f}-{:.1f})".format(age_median, age_iqr_l, age_iqr_u))
    print("Overall average BMI: {:.1f} ({:.1f})".format(bmi_mean, bmi_std))
    print("Overall median BMI: {:.1f} ({:.1f}-{:.1f})".format(bmi_median, bmi_iqr_l, bmi_iqr_u))
    print("\nRace counts:")
    print(race_counts)
    print("\nEthnicity counts:")
    print(ethnicity_counts)

    # Show counts for INFSTAT and SYMSTAT columns
    infstat_counts = df['INFSTAT'].value_counts()
    symstat_counts = df['SYMSTAT'].value_counts()
    detail_counts = df['INFSYM'].value_counts()

    print(f"\nClinical status for {group}:")
    print("Counts by INFSTAT:")
    print(infstat_counts)
    print("\nCounts by SYMSTAT:")
    print(symstat_counts)
    print("\nCounts by INFSYM:")
    print(detail_counts)

    # show counts for Vaccine
    vaccine_counts = df['Vaccine'].value_counts()
    print("\nCounts by Vaccine:")
    print(vaccine_counts)

## 3. Normalize HR against the participant baseline

For each participant, calculate `(HR - baseline mean) / baseline SD` using all
retained observations on relative days `< 0`. 
Missing or zero baseline SD can yield missing or nonfinite normalized values;
this step does not impute them.

Also calculate elapsed hours from the exact inoculation timestamp and extract
clock time. Create cohort-specific views with a shared set of HR columns for the
following comparisons. Timestamps must use compatible time zones.

In [ ]:
def z_score_hr(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Normalize heart rate data for each patient by calculating the z-score based on their baseline heart rate.
    '''
    full_df = pd.DataFrame()
    for patient in df['Participant ID'].unique():
        patient_df = df.loc[df['Participant ID'] == patient].copy()
        inoculation_time = test_info[test_info['Participant ID'] == patient]['Inoculation Time'].values[0]
        inoculation_datetime = pd.to_datetime(inoculation_time)
        patient_df['relative_hour_to_inoculation'] = (patient_df['dateTime'] - inoculation_datetime).dt.total_seconds() / 3600  # convert to hours
        mean_baseline_hr = patient_df[patient_df['relative_day_to_inoculation'] < 0]['hr'].mean()
        std_baseline_hr = patient_df[patient_df['relative_day_to_inoculation'] < 0]['hr'].std()
        patient_df['normalized_hr'] = (patient_df['hr'] - mean_baseline_hr )/ std_baseline_hr
        if full_df.empty:
            full_df = patient_df
        else:
            full_df = pd.concat([full_df, patient_df], ignore_index=True)
    full_df['time'] = full_df['dateTime'].dt.time
    return full_df

In [ ]:
# Keep the same measurement columns in each daily-analysis cohort.
HR_COLUMNS = [
    "Participant ID", "time", "hr", "normalized_hr", "relative_day_to_inoculation"
]


def select_hr_cohort(
    measurements: pd.DataFrame, participant_ids, reset_index: bool = True
) -> pd.DataFrame:
    """Copy the standard daily HR columns for the requested participants."""
    selected = measurements.loc[
        measurements["Participant ID"].isin(participant_ids), HR_COLUMNS
    ].copy()
    return selected.reset_index(drop=True) if reset_index else selected


final_hr_df_normalized = z_score_hr(final_hr_df)
inoculated_ids = final_hr_df_normalized.loc[
    ~final_hr_df_normalized["Participant ID"].isin(shams_id), "Participant ID"
]
inoculated_df = select_hr_cohort(final_hr_df_normalized, inoculated_ids)
infected_df = select_hr_cohort(final_hr_df_normalized, infected["Participant ID"])
uninfected_df = select_hr_cohort(final_hr_df_normalized, uninfected["Participant ID"])
sham_df = select_hr_cohort(final_hr_df_normalized, shams_id)

SI_hr_df_final = select_hr_cohort(final_hr_df_normalized, SI["Participant ID"], False)
AI_hr_df_final = select_hr_cohort(final_hr_df_normalized, AI["Participant ID"], False)
SU_hr_df_final = select_hr_cohort(final_hr_df_normalized, SU["Participant ID"], False)
AU_hr_df_final = select_hr_cohort(final_hr_df_normalized, AU["Participant ID"], False)

## 4. Compare repeated measurements in the SI cohort

Calculate each SI participant's mean normalized HR during 08:00-20:00 (daytime) and during
00:00-08:00 (nighttime, assume main long sleep) on the **following** day. 
Both endpoints are inclusive. Pool all negative days into a baseline labeled -1; 
baseline nights use negative days themselves.
Compute the daytime minus nighttime difference, then drop rows missing any metric.

For each metric, run a one-way repeated-measures ANOVA and paired day comparisons
through day 6. Holm adjustment covers all day pairs separately within each metric.
The heatmap shows only adjusted p-values ≤ 0.05.

In [ ]:
hr_df = pd.DataFrame()

for participant in SI_hr_df_final['Participant ID'].unique():
    participant_df = SI_hr_df_final[
        SI_hr_df_final['Participant ID'] == participant
    ].copy()

    # Group all pre-inoculation days into a single label (-1)
    participant_df['day_group'] = participant_df['relative_day_to_inoculation']
    participant_df.loc[participant_df['day_group'] < 0, 'day_group'] = -1

    for day in sorted(participant_df['day_group'].unique()):

        if day == -1:
            # Use all daytime data before inoculation as baseline
            day_df = participant_df[
                (participant_df['relative_day_to_inoculation'] < 0) &
                (participant_df['time'] >= pd.to_datetime('08:00:00').time()) &
                (participant_df['time'] <= pd.to_datetime('20:00:00').time())
            ]

            # Use all nighttime data before inoculation as baseline
            night_df = participant_df[
                (participant_df['relative_day_to_inoculation'] < 0) &
                (participant_df['time'] >= pd.to_datetime('00:00:00').time()) &
                (participant_df['time'] <= pd.to_datetime('08:00:00').time())
            ]

        else:
            day_df = participant_df[
                (participant_df['relative_day_to_inoculation'] == day) &
                (participant_df['time'] >= pd.to_datetime('08:00:00').time()) &
                (participant_df['time'] <= pd.to_datetime('20:00:00').time())
            ]

            night_df = participant_df[
                (participant_df['relative_day_to_inoculation'] == day + 1) &
                (participant_df['time'] >= pd.to_datetime('00:00:00').time()) &
                (participant_df['time'] <= pd.to_datetime('08:00:00').time())
            ]

        day_hr_mean = day_df['normalized_hr'].mean()
        night_hr_mean = night_df['normalized_hr'].mean()

        hr_df = pd.concat([
            hr_df,
            pd.DataFrame({
                'participant_id': [participant],
                'relative_day_to_inoculation': [day],
                'daytime_hr_mean': [day_hr_mean],
                'nighttime_hr_mean': [night_hr_mean],
                'delta_hr': [day_hr_mean - night_hr_mean]
            })
        ], ignore_index=True)
hr_df = hr_df.dropna().reset_index(drop=True)

In [ ]:
def pairwise_t_tests(hr_df: pd.DataFrame, metric: str) -> dict:
    '''
    Perform repeated measures ANOVA and pairwise t-tests on the specified heart rate metric.
    The function filters the data to include only days up to 6 days post-inoculation,
    then conducts a repeated measures ANOVA and pairwise t-tests with Holm correction for multiple comparisons.
    '''
    hr_df = hr_df[hr_df['relative_day_to_inoculation'] <= 6].reset_index(drop=True)
    anova = pg.rm_anova(
        data=hr_df,
        dv=metric,
        within="relative_day_to_inoculation",
        subject="participant_id",
        detailed=True
    )

    pairwise = pg.pairwise_tests(
        data=hr_df,
        dv=metric,
        within="relative_day_to_inoculation",
        subject="participant_id",
        padjust="holm",   # Adjust across day pairs within this metric.
        effsize="cohen"
    )

    d1_d2_comparisons = pairwise[
        ((pairwise["A"] == 1) | (pairwise["B"] == 1) |
        (pairwise["A"] == 2) | (pairwise["B"] == 2))
    ]

    # Keep the focused day 1/day 2 comparisons alongside the full test table.
    d1_d2_comparisons = d1_d2_comparisons[[
        "A", "B", "T", "dof", "p-corr", "cohen"
    ]]

    days = sorted(hr_df["relative_day_to_inoculation"].unique())

    # initialize matrix
    p_matrix = pd.DataFrame(
        np.ones((len(days), len(days))),
        index=days,
        columns=days
    )

    # fill matrix
    for _, row in pairwise.iterrows():
        a, b = row["A"], row["B"]
        p = row["p-corr"]
        p_matrix.loc[a, b] = p
        p_matrix.loc[b, a] = p

    masked_p = np.ma.masked_where(p_matrix.values > 0.05, p_matrix.values)

    plt.figure(figsize=(7, 6))

    im = plt.imshow(masked_p, aspect="auto", origin="lower", vmin=0, vmax=0.05)

    plt.colorbar(im, label="Adjusted p-value", ticks=np.arange(0, 0.051, 0.01))

    plt.xticks(range(len(days)), days)
    plt.yticks(range(len(days)), days)

    plt.xlabel("Relative Day to Inoculation")
    plt.ylabel("Relative Day to Inoculation")
    if metric == "daytime_hr_mean":
        plt.title(f"Diurnal Heart Rate", loc='left')
    elif metric == "nighttime_hr_mean":
        plt.title(f"Nocturnal Heart Rate", loc='left')
    elif metric == "delta_hr":
        plt.title(f"Diurnal-Nocturnal Heart Rate", loc='left')

    plt.show()

    return {
        "anova": anova,
        "pairwise": pairwise,
        "day_1_2_comparisons": d1_d2_comparisons,
        "adjusted_p_values": p_matrix,
    }

In [ ]:
statistical_results = {}
for metric in ["daytime_hr_mean", "nighttime_hr_mean", "delta_hr"]:
    results = pairwise_t_tests(hr_df, metric)
    statistical_results[metric] = results

## 5. Summarize raw daytime and nighttime HR across cohorts

Within each participant and day, average raw HR at each observed clock time, then
average those clock-time means within daytime (08:00-20:00) and the following
night (00:00-08:00). Negative-day summaries are averaged into one baseline at day -1.
Only daytime labels through day 6 are included.

Next calculate cohort means and between-participant SDs. Bar plots compare
sham, SI, and inoculated uninfected participants; error bars show SD, not confidence
intervals. 

In [ ]:
def cal_hr(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Calculate daytime and nighttime heart rate means and standard deviations for each day.
    '''
    hr_df = pd.DataFrame()
    for participant in df['Participant ID'].unique():
        participant_df = df[df['Participant ID'] == participant].drop(columns=['Participant ID'])
        for day in participant_df['relative_day_to_inoculation'].unique():
            if day > 6:
                continue
            day_time = day
            # the night following the day
            night_time = day+1
            day_df = participant_df[(participant_df['relative_day_to_inoculation'] == day_time)
                        & (participant_df['time'] >= pd.to_datetime('08:00:00').time())
                        & (participant_df['time'] <= pd.to_datetime('20:00:00').time())]
            night_df = participant_df[(participant_df['relative_day_to_inoculation'] == night_time)
                          & ((participant_df['time'] >= pd.to_datetime('00:00:00').time())
                             & (participant_df['time'] <= pd.to_datetime('08:00:00').time()))]
            day_hr_mean = day_df.groupby('time')['hr'].mean().mean()
            night_hr_mean = night_df.groupby('time')['hr'].mean().mean()
            hr_df = pd.concat([hr_df, pd.DataFrame({'participant': participant,
                                                    'relative_day_to_inoculation':[day_time],
                                                    'daytime_hr_mean':[day_hr_mean],
                                                    'nighttime_hr_mean':[night_hr_mean],
                                                    'delta_hr':[day_hr_mean-night_hr_mean]})],
                                                    ignore_index=True)
            hr_df.loc[hr_df['relative_day_to_inoculation'] < 0, 'relative_day_to_inoculation'] = -1
    hr_df = hr_df.groupby(['participant', 'relative_day_to_inoculation']).mean().reset_index()
    return hr_df

def cal_hr_std(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Calculate standard deviations for heart rate metrics.
    '''
    calculated_df = df.drop(columns='participant').groupby('relative_day_to_inoculation').agg(
        daytime_hr_mean=('daytime_hr_mean', 'mean'),
        daytime_hr_std=('daytime_hr_mean', 'std'),
        nighttime_hr_mean=('nighttime_hr_mean', 'mean'),
        nighttime_hr_std=('nighttime_hr_mean', 'std'),
        diff_mean=('delta_hr', 'mean'),
        diff_std=('delta_hr', 'std')
    ).reset_index()
    return calculated_df

def plot_hr_comparison(
    hr_type: str, sham_hr_summary: pd.DataFrame,
    SI_hr_summary: pd.DataFrame, uninfected_hr_summary: pd.DataFrame,
) -> None:
    '''
    Plot the comparison of heart rate metrics across different groups.
    '''
    if hr_type == 'daytime':
        metric = 'daytime_hr_mean'
        y_err = 'daytime_hr_std'
        ylabel = 'Diurnal Heart Rate (bpm)'
        title = 'Diurnal Heart Rate'
        ylim = (40, 110)
    elif hr_type == 'nighttime':
        metric = 'nighttime_hr_mean'
        y_err = 'nighttime_hr_std'
        ylabel = 'Nocturnal Heart Rate (bpm)'
        title = 'Nocturnal Heart Rate'
        ylim = (40, 110)
    elif hr_type == 'delta':
        metric = 'diff_mean'
        y_err = 'diff_std'
        ylabel = 'Diurnal-Nocturnal Heart Rate (bpm)'
        title = 'Diurnal-Nocturnal Heart Rate'
        ylim = (-10, 25)
    else:
        raise ValueError("hr_type must be daytime, nighttime, or delta.")
    plt.figure(figsize=(12, 6))
    # Bar positions
    x_sham = sham_hr_summary['relative_day_to_inoculation']
    x_inf = SI_hr_summary['relative_day_to_inoculation']
    x_uninf = uninfected_hr_summary['relative_day_to_inoculation']

    bar_width = 0.25

    # Sham bars
    plt.bar(
        x_sham - bar_width,
        sham_hr_summary[metric],
        yerr=sham_hr_summary[y_err],
        width=bar_width,
        label='Sham (N=7)',
        capsize=4
    )

    # Uninfected bars
    plt.bar(
        x_uninf,
        uninfected_hr_summary[metric],
        yerr=uninfected_hr_summary[y_err],
        width=bar_width,
        label='Inoculated but Uninfected (N=16)',
        capsize=4
    )

    # Infected bars
    plt.bar(
        x_inf + bar_width,
        SI_hr_summary[metric],
        yerr=SI_hr_summary[y_err],
        width=bar_width,
        label='Symptomatic and Infected (N=46)',
        capsize=4,
        color='red'
    )

    plt.xlabel('Relative Day to Inoculation', fontsize=24)
    plt.xticks(fontsize=24)
    plt.yticks(fontsize=24)
    plt.ylim(ylim)
    plt.ylabel(ylabel, fontsize=24)
    plt.title(title, fontsize=28, loc='left')
    plt.legend(fontsize=16)
    plt.show()

In [ ]:
infected_hr_summary = cal_hr(infected_df)
uninfected_hr_summary = cal_hr(uninfected_df)
sham_hr_summary = cal_hr(sham_df)
SI_hr_summary = cal_hr(SI_hr_df_final)

infected_hr_summary = cal_hr_std(infected_hr_summary)
uninfected_hr_summary = cal_hr_std(uninfected_hr_summary)
sham_hr_summary = cal_hr_std(sham_hr_summary)
SI_hr_summary = cal_hr_std(SI_hr_summary)

In [ ]:
plot_hr_comparison('daytime', sham_hr_summary, SI_hr_summary, uninfected_hr_summary)
plot_hr_comparison('nighttime', sham_hr_summary, SI_hr_summary, uninfected_hr_summary)
plot_hr_comparison('delta', sham_hr_summary, SI_hr_summary, uninfected_hr_summary)

## 6. Visualize clock-time profiles and continuous HR

### 6.1 Compare daily clock-time profiles

Average raw HR across participants at each clock time and align observations to a
15-second grid. Smooth each daily curve with a Gaussian filter.
Missing grid values are left missing and can propagate through smoothing; the two
sampling frequencies can therefore contribute differently.

Display days -2 through 7, limiting the observed data on day -2 to noon (check-in) onward and
day 7 to noon (check-out) or earlier. Omit SU day 7 for low number of observations. 
The reference line uses all retained negative days plus day 0 through noon.

In [ ]:
# Generate all possible times in a day at 15-second intervals
all_times = pd.date_range("2000-01-01 00:00:00", "2000-01-01 23:59:45", freq="15s").time
all_times_df = pd.DataFrame({'time': all_times.astype(str)})
all_times_df['time'] = pd.to_datetime(all_times_df['time'], format='%H:%M:%S').dt.time

In [ ]:
dfs = [SI_hr_df_final, AU_hr_df_final, SU_hr_df_final, AI_hr_df_final]
categories = ['Symptomatic and Infected (N=46)', 'Asymptomatic and Uninfected (N=17)',
              'Symptomatic and Uninfected (N=6)', 'Asymptomatic and Infected (N=15)']
title_labels = [r'$\bf{A.}$ Symptomatic and Infected (N=46)',
    r'$\bf{B.}$ Asymptomatic and Uninfected (N=17)',
    r'$\bf{C.}$ Symptomatic and Uninfected (N=6)',
    r'$\bf{D.}$ Asymptomatic and Infected (N=15)']
fig, axes = plt.subplots(2, 2, figsize=(20, 12))

for ax, df, category, title_label in zip(axes.flat, dfs, categories, title_labels):
    df = df.drop(columns=['Participant ID'])
    unique_days = sorted(df['relative_day_to_inoculation'].unique())
    unique_days = [d for d in unique_days if d >= -2 and d <= 7]
    pos_days = [d for d in unique_days if d > 0]

    rainbow_cmap = cm.get_cmap('turbo_r', len(pos_days))

    pos_color_map = {day: rainbow_cmap(i)
                    for i, day in enumerate(sorted(pos_days))}

    baseline_df = pd.concat([df[df['relative_day_to_inoculation'] < 0],
        df[(df['relative_day_to_inoculation'] == 0) & (df['time'] <= pd.to_datetime('12:00:00').time())]])
    baseline_hr = baseline_df['hr'].mean()
    for day in unique_days:
        if category == 'Symptomatic and Uninfected (N=6)' and day == 7:
            continue
        day_df = df[df['relative_day_to_inoculation'] == day].copy()
        day_df = day_df.drop(columns=['relative_day_to_inoculation'])
        day_df = day_df.groupby('time').mean().reset_index()
        if day == -2:
            day_df = day_df[day_df['time'] >= pd.to_datetime('12:00:00').time()]
        elif day == 7:
            day_df = day_df[day_df['time'] <= pd.to_datetime('12:00:00').time()]
        if len(day_df) != len(all_times_df):
            day_df = all_times_df.merge(day_df, on='time', how='outer')

        x = range(len(day_df))
        smoothed_hr = gaussian_filter1d(day_df['hr'], sigma=50)  # apply Gaussian smoothing

        if day <= 0:
            color = 'lightblue'
        else:
            color = pos_color_map[day]
        if day == -2:
            line, = ax.plot(x, smoothed_hr, label=f"Day {int(day)}",
                            linewidth=3, color=color,
                            alpha=0.5)
        elif day == 7:
            line, = ax.plot(x, smoothed_hr, label=f"Day {int(day)}",
                            linewidth=3, color='gray', alpha=0.5)
        else:
            line, = ax.plot(x, smoothed_hr, label=f"Day {int(day)}", linewidth=3, color=color)

    ax.axhline(baseline_hr, color='black', linestyle='--', label='Mean Baseline HR')
    tick_indices = range(0, len(day_df), max(1, len(day_df) // 12))
    ax.set_xticks(ticks=tick_indices)
    ax.set_xticklabels([str(day_df['time'].iloc[i])[:5] for i in tick_indices],
        rotation=45,
        fontsize=22)
    ax.set_xlabel("Time", fontsize=24)
    ax.set_ylim([55, 95])
    ax.set_ylabel("Raw Heart Rate (bpm)", fontsize=24)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{int(y)}'))
    ax.tick_params(axis='y', labelsize=22)
    ax.set_title(title_label, fontsize=24, loc='left')

handles, labels = axes[0, 0].get_legend_handles_labels()

# Put legend in the last subplot
fig.legend(
    handles,
    labels,
    title='Relative Day to Inoculation',
    title_fontsize=18,
    loc='center right',          # middle right
    bbox_to_anchor=(1.09, 0.5),  # push legend outside
    fontsize=18
)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

### 6.2 Compare continuous normalized HR trajectories

Restrict observations to -48 through +144 hours relative to inoculation and average 
by **exact elapsed-hour values**, without hourly binning. Plot the selected cohort alongside
sham participants. The curves are gaussian smoothed.

In [ ]:
full_df = final_hr_df_normalized[(final_hr_df_normalized['relative_hour_to_inoculation'] >= -48) & (final_hr_df_normalized['relative_hour_to_inoculation'] <= 144)].reset_index(drop=True)
continuous_inoculated_df = full_df[~full_df['Participant ID'].isin(shams_id)].reset_index(drop=True)[['relative_hour_to_inoculation',
    'normalized_hr',
    'hr']]
continuous_FLUH_df = full_df[full_df['Participant ID'].isin(FLUH['Participant ID'])].reset_index(drop=True)[['relative_hour_to_inoculation',
    'normalized_hr']]
continuous_FLUM_df = full_df[full_df['Participant ID'].isin(FLUM['Participant ID'])].reset_index(drop=True)[['relative_hour_to_inoculation',
    'normalized_hr']]
continuous_FLUL_df = full_df[full_df['Participant ID'].isin(FLUL['Participant ID'])].reset_index(drop=True)[['relative_hour_to_inoculation',
    'normalized_hr']]
continuous_sham_df = full_df[full_df['Participant ID'].isin(shams_id)].reset_index(drop=True)[['relative_hour_to_inoculation',
    'normalized_hr',
    'hr']]
continuous_infected_df = full_df[full_df['Participant ID'].isin(infected['Participant ID'])].reset_index(drop=True)[['relative_hour_to_inoculation',
    'normalized_hr',
    'hr']]
continuous_uninfected_df = full_df[full_df['Participant ID'].isin(uninfected['Participant ID'])].reset_index(drop=True)[['relative_hour_to_inoculation',
    'normalized_hr',
    'hr']]
continuous_SI_df = full_df[full_df['Participant ID'].isin(SI['Participant ID'])].reset_index(drop=True)[['relative_hour_to_inoculation',
    'normalized_hr',
    'hr']]

In [ ]:
continuous_inoculated_df = continuous_inoculated_df.groupby('relative_hour_to_inoculation').mean()
continuous_infected_df = continuous_infected_df.groupby('relative_hour_to_inoculation').mean()
continuous_uninfected_df = continuous_uninfected_df.groupby('relative_hour_to_inoculation').mean()
continuous_FLUH_df = continuous_FLUH_df.groupby('relative_hour_to_inoculation').mean()
continuous_FLUM_df = continuous_FLUM_df.groupby('relative_hour_to_inoculation').mean()
continuous_FLUL_df = continuous_FLUL_df.groupby('relative_hour_to_inoculation').mean()
continuous_sham_df = continuous_sham_df.groupby('relative_hour_to_inoculation').mean()
continuous_SI_df = continuous_SI_df.groupby('relative_hour_to_inoculation').mean()

In [ ]:
def plot_continuous_hr(dataframe: str) -> None:
    '''
    Plot continuous heart rate data for a specified group of participants.
    '''
    if dataframe == 'full':
        df = continuous_inoculated_df.copy()
        group = 'Inoculated (N=77)'
    elif dataframe == 'infected':
        df = continuous_infected_df.copy()
        group = 'Infected (N=61)'
    elif dataframe == 'uninfected':
        df = continuous_uninfected_df.copy()
        group = 'Uninfected (N=16)'
    elif dataframe == 'SI':
        df = continuous_SI_df.copy()
        group = 'Symptomatic Infected (N=46)'
    else:
        raise ValueError("Invalid dataframe specified.")

    x_ticks_positions = np.arange(-48, 145, 12)

    plt.figure(figsize=(12, 3))
    plt.plot(continuous_sham_df.index,
        gaussian_filter1d(continuous_sham_df['normalized_hr'],
        sigma=50),
        label='Shams (N=7)',
        color='#708090')
    plt.plot(df.index, gaussian_filter1d(df['normalized_hr'], sigma=10), label=group, color='red')
    plt.xlabel('Relative Hour to Inoculation', fontsize=12)
    plt.ylabel('Normalized Heart Rate', fontsize=12)
    plt.title(r'$\bf{A.}$ Continuous Heart Rate', fontsize=14, loc='left')
    plt.ylim(-1.25, 2.75)
    plt.yticks(np.arange(-1, 3, 1))

    # Set the x-axis ticks
    plt.xticks(x_ticks_positions)

    # Large legend below the plot
    plt.legend(
        loc='upper left',
    )

    plt.show()

In [ ]:
plot_continuous_hr('SI')

## 7. Align nocturnal HR with viral load and symptom scores

Based on previous observations, we focused on sleep data from clock times ≥23:59 or
≤08:00. Baseline data is defined as relative hours before inoculation (relative hour < 0).
Average HR metrics by participant and relative calendar day, then
join clinical measurements on participant ID and day. Plot post-inoculation days
0 through 7 for each clinical cohort, with shared limits per metric.

Since the clinical assessments are only taken after inoculation, therefore, clinical assessment 
before the inoculation is missing after merging. Missing viral load is filled with 0.5; 
missing symptom score is filled with with 0.

In [ ]:
viral_loads = pd.read_csv(DATA_DIR / "viral_loads.csv")
symptom_scores = pd.read_csv(DATA_DIR / "symptom_scores.csv")

In [ ]:
# Select the existing clock-time interval spanning midnight: 23:59-08:00.
sleep_start = pd.to_datetime('23:59:00').time()
sleep_end = pd.to_datetime('08:00:00').time()

sleep_full_df = full_df[(full_df['time'] >= sleep_start) | (full_df['time'] <= sleep_end)].reset_index(drop=True)
baseline = (
    sleep_full_df[sleep_full_df['relative_hour_to_inoculation'] < 0]
    .groupby('Participant ID')['hr']
    .mean()
)
sleep_full_df['hr_delta'] = sleep_full_df['hr'] - sleep_full_df['Participant ID'].map(baseline)
sleep_full_df['delta_perc'] = (sleep_full_df['hr_delta'] / sleep_full_df['hr']) * 100
sleep_full_df = sleep_full_df[['Participant ID',
    'hr',
    'normalized_hr',
    'hr_delta',
    'delta_perc',
    'relative_day_to_inoculation']].groupby(['Participant ID',
    'relative_day_to_inoculation']).mean().reset_index()
sleep_full_df = sleep_full_df.merge(test_info[['Participant ID',
    'INFSYM',
    'Dosage']],
    on='Participant ID',
    how='left')

sleep_full_df = sleep_full_df[sleep_full_df['relative_day_to_inoculation'] >= -2].rename(columns={'Participant ID': 'Participant_ID',
    'relative_day_to_inoculation': 'Day',
    })
sleep_full_df = sleep_full_df.merge(viral_loads,
    on=['Participant_ID',
    'Day'],
    how='left').fillna(0.5)
sleep_full_df_t = sleep_full_df[sleep_full_df['Day']>=0].copy()
sleep_full_df_t = pd.merge(sleep_full_df_t,
    symptom_scores,
    on=['Participant_ID',
    'Day'],
    how='left').fillna(0.0)

In [ ]:
hr_metric = 'hr_delta'

# hr_metric = 'normalized_hr'
# hr_metric = 'delta_perc'

if hr_metric == 'delta_perc':
    label = 'HR Deviation (%)'
elif hr_metric == 'normalized_hr':
    label = 'Normalized HR'
else:
    label = 'HR Deviation (bpm)'

categories = [
    'Symptomatic and Infected',
    'Asymptomatic and Uninfected',
    'Asymptomatic and Infected',
    'Symptomatic and Uninfected',
]

title_labels = [
    r'$\bf{A.}$ Symptomatic and Infected (N=46)',
    r'$\bf{B.}$ Asymptomatic and Uninfected (N=17)',
    r'$\bf{C.}$ Asymptomatic and Infected (N=15)',
    r'$\bf{D.}$ Symptomatic and Uninfected (N=6)',
]

# Compute global y-limits across categories for each variable
hr_min, hr_max = float('inf'), float('-inf')
rna_min, rna_max = float('inf'), float('-inf')
sym_min, sym_max = float('inf'), float('-inf')

for cat in categories:
    sub = sleep_full_df_t[sleep_full_df_t['INFSYM'] == cat]

    if sub.empty:
        continue

    grouped = sub.groupby('Day').agg({
        hr_metric: 'mean',
        'Viral Load (copies per mL)': 'mean',
        'Symptom_Score': 'mean'
    }).reset_index()

    grouped = grouped[grouped['Day'] < 8]

    # Skip if grouped is empty
    if grouped.empty:
        continue

    hr_min = min(hr_min, grouped[hr_metric].min())
    hr_max = max(hr_max, grouped[hr_metric].max())

    rna_min = min(rna_min, grouped['Viral Load (copies per mL)'].min())
    rna_max = max(rna_max, grouped['Viral Load (copies per mL)'].max())

    sym_min = min(sym_min, grouped['Symptom_Score'].min())
    sym_max = max(sym_max, grouped['Symptom_Score'].max())


# In case one of the variables was not present at all, provide sensible defaults
if hr_min == float('inf'):
    hr_min, hr_max = 0, 1

if rna_min == float('inf'):
    rna_min, rna_max = 0, 1

if sym_min == float('inf'):
    sym_min, sym_max = 0, 1


hr_padding = (hr_max - hr_min) * 0.1 if hr_max != hr_min else 0.1 * max(1, abs(hr_max))
rna_padding = (rna_max - rna_min) * 0.1 if rna_max != rna_min else 0.1 * max(1, abs(rna_max))
sym_padding = (sym_max - sym_min) * 0.1 if sym_max != sym_min else 0.1 * max(1, abs(sym_max))

hr_ylim = (hr_min - hr_padding, hr_max + hr_padding)
rna_ylim = (rna_min - rna_padding, rna_max + rna_padding)
sym_ylim = (sym_min - sym_padding, sym_max + sym_padding)


# Plotting
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=False)

for idx, (ax, cat, title_label) in enumerate(zip(axes.flat, categories, title_labels)):
    sub = sleep_full_df_t[sleep_full_df_t['INFSYM'] == cat]

    grouped = sub.groupby('Day').agg({
        hr_metric: 'mean',
        'Viral Load (copies per mL)': 'mean',
        'Symptom_Score': 'mean'
    }).reset_index()

    grouped = grouped[grouped['Day'] < 8]

    # Add customized title
    ax.set_title(title_label, fontsize=16, loc='left',)

    # If there's no data for this category, show a message and skip plotting
    if grouped.empty:
        ax.text(
            0.5, 0.5,
            f'No data for\n{cat}',
            ha='center',
            va='center',
            fontsize=12,
            transform=ax.transAxes
        )

        ax.set_xlabel('Relative Day to Inoculation', fontsize=14)
        ax.tick_params(axis='x', labelsize=12)
        ax.tick_params(axis='y', labelsize=12)
        continue

    # Primary axis: HR metric
    p1, = ax.plot(
        grouped['Day'],
        grouped[hr_metric],
        marker='o',
        linestyle='-',
        color='mediumorchid',
        label=label
    )

    ax.set_ylabel(label, color='mediumorchid', fontsize=14)
    ax.tick_params(axis='y', labelcolor='mediumorchid', labelsize=12)
    ax.set_ylim(hr_ylim)

    ax.set_xlabel('Relative Day to Inoculation', fontsize=14)
    ax.tick_params(axis='x', labelsize=12)
    ax.grid(True, alpha=0.3)

    # Secondary axis: Viral load
    ax2 = ax.twinx()

    p2, = ax2.plot(
        grouped['Day'],
        grouped['Viral Load (copies per mL)'],
        marker='s',
        linestyle='--',
        color='darkorange',
        label='Viral load (copies/mL)'
    )

    ax2.set_ylabel('Viral load (copies/mL)', color='darkorange', fontsize=14)
    ax2.tick_params(axis='y', labelcolor='darkorange', labelsize=12)
    ax2.set_ylim(rna_ylim)

    # Tertiary axis: Symptom score
    ax3 = ax.twinx()

    # Move the third axis outwards so it does not overlap with ax2
    ax3.spines["right"].set_position(("outward", 60))
    ax3.spines["right"].set_visible(True)

    p3, = ax3.plot(
        grouped['Day'],
        grouped['Symptom_Score'],
        marker='D',
        linestyle=':',
        color='black',
        label='Symptom score'
    )

    ax3.set_ylabel('Symptom score', color='black', fontsize=14)
    ax3.tick_params(axis='y', labelcolor='black', labelsize=12)
    ax3.set_ylim(sym_ylim)


plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

## 8. Summarize physiology within recorded sleep windows

### 8.1 Align wearable HR to participant-specific sleep intervals

Read Oura `long_sleep` (main sleep) records through day 7. Missing files and 
participants with no HR data or only nonnegative sleep days are skipped. 
For multiple sleep records on a relative day, use the first record in file order. 
The sleep record's `day` determines the relative-day label; HR is selected 
inclusively between its bedtime start and end. All timestamps must already use 
compatible clocks and time zones. 

Normalize HR against all available HR at elapsed hours `< 0`, which include all
possible measurements before inoculation. Then calculate sleep HR deviations 
relative to baseline sleep windows. Keep sleep summaries from day -2
onward, alongside breath rate, HRV, and temperature deviation.

In [ ]:
sleep_results = []
for participant in test_info["Participant ID"].unique():
    participant_data = final_hr_df.loc[
        final_hr_df["Participant ID"] == participant
    ].dropna(subset=["hr"]).reset_index(drop=True)
    if participant_data.empty:
        continue
    inoculation_time = test_info.loc[
        test_info["Participant ID"] == participant, "Inoculation Time"
    ].iloc[0]
    inoculation_datetime = pd.to_datetime(inoculation_time)
    inoculation_date = inoculation_datetime.date()

    # Load the device's main sleep records, keeping their original file order.
    try:
        if participant.startswith("C"):
            sleep_df = pd.read_parquet(CIVICS_SLEEP_DIR / participant / "Sleep.parquet")
        else:
            sleep_df = pd.read_csv(EXHALE_SLEEP_DIR / participant / "sleep.csv")
    except FileNotFoundError:
        continue
    sleep_columns = [
        "average_breath", "average_heart_rate", "average_hrv", "day",
        "bedtime_start", "bedtime_end", "readiness_temperature_deviation",
    ]
    sleep_df = sleep_df.loc[sleep_df["type"] == "long_sleep", sleep_columns].copy()
    sleep_df["day"] = pd.to_datetime(sleep_df["day"])
    sleep_df["relative_day_to_inoculation"] = (
        sleep_df["day"].dt.normalize() - pd.Timestamp(inoculation_date)
    ).dt.days
    for column in ["bedtime_start", "bedtime_end"]:
        sleep_df[column] = pd.to_datetime(sleep_df[column])
    sleep_df = sleep_df.loc[sleep_df["relative_day_to_inoculation"] <= 7]
    if sleep_df.empty or sleep_df["relative_day_to_inoculation"].min() >= 0:
        continue

    # Standardize against all HR before the exact inoculation timestamp.
    participant_data["relative_hour_to_inoculation"] = (
        participant_data["dateTime"] - inoculation_datetime
    ).dt.total_seconds() / 3600
    baseline_hr = participant_data.loc[
        participant_data["relative_hour_to_inoculation"] < 0, "hr"
    ]
    participant_data["normalized_hr"] = (
        participant_data["hr"] - baseline_hr.mean()
    ) / baseline_hr.std()
    participant_data["datetime"] = [
        datetime.combine(inoculation_date + timedelta(days=int(day)), clock_time)
        for day, clock_time in zip(
            participant_data["relative_day_to_inoculation"],
            participant_data["dateTime"].dt.time,
        )
    ]

    # Preserve the first sleep interval per day, including for baseline windows.
    daily_sleep = sleep_df.drop_duplicates("relative_day_to_inoculation", keep="first")
    baseline_windows = []
    for _, sleep_record in daily_sleep.iterrows():
        if sleep_record["relative_day_to_inoculation"] >= 0:
            continue
        in_sleep = participant_data["datetime"].between(
            sleep_record["bedtime_start"], sleep_record["bedtime_end"]
        )
        baseline_windows.append(participant_data.loc[in_sleep])
    baseline_sleep = pd.concat(baseline_windows, ignore_index=True)

    # Summarize HR within each inclusive bedtime interval, then attach device metrics.
    for _, sleep_record in daily_sleep.iterrows():
        day = sleep_record["relative_day_to_inoculation"]
        if day < -2:
            continue
        in_sleep = participant_data["datetime"].between(
            sleep_record["bedtime_start"], sleep_record["bedtime_end"]
        )
        sleep_data = participant_data.loc[in_sleep]
        mean_hr = sleep_data["hr"].mean()
        mean_normalized_hr = sleep_data["normalized_hr"].mean()
        sleep_results.append({
            "Participant_ID": participant,
            "Day": day,
            "HR_Deviation": mean_hr - baseline_sleep["hr"].mean(),
            "Normalized_HR_Deviation": (
                mean_normalized_hr - baseline_sleep["normalized_hr"].mean()
            ),
            "HR": mean_hr,
            "Normalized_HR": mean_normalized_hr,
            "Average_Breath": sleep_record["average_breath"],
            "Average_HRV": sleep_record["average_hrv"],
            "Average_Temperature_Deviation": sleep_record["readiness_temperature_deviation"],
        })

if not sleep_results:
    raise ValueError("No eligible sleep summaries; check sleep paths and baseline coverage.")
final_results = pd.DataFrame(sleep_results)

### 8.2 Calculate deviations, attach cohort labels, and plot

To account for individual variability, we use deviation relative to baseline to
describe physiological changes due to infection. Since temperature has already
been reported as deviation to baseline, and HR deviation has been manually calcualted,
for breath rate and HRV, subtract each participant's mean over available `Day ≤ 0`
rows in `final_results` (days -2, -1, and 0 when present), then round to two decimals.
Day 0 is included as baseline because it reprensents the sleep data ending on day 0.

Assign the four clinical groups in SI, AI, SU, AU precedence order. Relabel
asymptomatic uninfected sham participants as `Sham`; these rows are excluded from
the final four-cohort figure.

In [ ]:
for participant in final_results['Participant_ID'].unique():
    participant_data = final_results[final_results['Participant_ID'] == participant]
    baseline_df = participant_data[participant_data['Day'] <= 0]

    for metrics in ['Average_Breath', 'Average_HRV']:
        diff = (participant_data[metrics] - baseline_df[metrics].mean()).round(2)

        final_results.loc[
            final_results['Participant_ID'] == participant,
            f'{metrics}_Deviation'
        ] = diff

In [ ]:
# Use explicit .loc assignment so labels update reliably across pandas versions.
final_results["INFSYM"] = pd.Series(index=final_results.index, dtype="object")
cohort_labels = [
    (SI, "Symptomatic and Infected"),
    (AI, "Asymptomatic and Infected"),
    (SU, "Symptomatic and Uninfected"),
    (AU, "Asymptomatic and Uninfected"),
]
for cohort, label in cohort_labels:
    matches = final_results["Participant_ID"].isin(cohort["Participant ID"])
    final_results.loc[matches & final_results["INFSYM"].isna(), "INFSYM"] = label

In [ ]:
vaccine = test_info[['Participant ID',
    'Dosage Name']].rename(columns={'Participant ID': 'Participant_ID'})
final_results = final_results.merge(vaccine, on='Participant_ID', how='left')

In [ ]:
sham_mask = (
    final_results["Dosage Name"].eq("SHAM")
    & final_results["INFSYM"].eq("Asymptomatic and Uninfected")
)
final_results.loc[sham_mask, "INFSYM"] = "Sham"

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

metrics = [
    ('HR_Deviation', 'Heart Rate (bpm)', 'A.'),
    ('Average_Breath_Deviation', 'Breath Rate (bpm)', 'B.'),
    ('Average_HRV_Deviation', 'Heart Rate Variability (ms)', 'C.'),
    ('Average_Temperature_Deviation', 'Temperature (°C)', 'D.')
]

categories = [
    'Symptomatic and Infected',
    'Asymptomatic and Infected',
    'Symptomatic and Uninfected',
    'Asymptomatic and Uninfected'
]

category_colors = {
    'Symptomatic and Infected': '#D55E00',
    'Asymptomatic and Infected': '#E69F00',
    'Symptomatic and Uninfected': '#0072B2',
    'Asymptomatic and Uninfected': '#009E73',
    'Uninfected': 'purple'
}

for ax, (metric, ylabel, title) in zip(axes.flat, metrics):

    for cat in categories:

        if cat == 'Uninfected':
            sub = final_results[
                final_results['INFSYM'].str.contains('Uninfected', na=False)
            ]
        else:
            sub = final_results[
                final_results['INFSYM'] == cat
            ]

        sub = (
            sub[['Day', metric]]
            .groupby('Day', as_index=False)
            .mean()
        )

        ax.plot(
            sub['Day'],
            sub[metric],
            marker='o',
            label=cat,
            alpha=0.8,
            linewidth=2,
            color=category_colors[cat]
        )

    ax.set_title(
        title,
        fontsize=24,
        fontweight='bold',
        loc='left',
        pad=10
    )

    ax.set_xlabel(
        'Relative Day to Inoculation',
        fontsize=22,
        labelpad=8
    )

    ax.set_ylabel(
        ylabel,
        fontsize=22,
        labelpad=10
    )

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=18,
        pad=5
    )

    ax.grid(True, alpha=0.3)


legend_elements = [
    Line2D(
        [0], [0],
        marker='o',
        linestyle='None',
        color='#D55E00',
        markersize=8,
        label='Symptomatic and Infected (N=43)'
    ),
    Line2D(
        [0], [0],
        marker='o',
        linestyle='None',
        color='#E69F00',
        markersize=8,
        label='Asymptomatic and Infected (N=14)'
    ),
    Line2D(
        [0], [0],
        marker='o',
        linestyle='None',
        color='#0072B2',
        markersize=8,
        label='Symptomatic and Uninfected (N=3)'
    ),
    Line2D(
        [0], [0],
        marker='o',
        linestyle='None',
        color='#009E73',
        markersize=8,
        label='Asymptomatic and Uninfected (N=12)'
    ),
]


fig.legend(
    handles=legend_elements,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.99),
    ncol=2,
    fontsize=16,
    frameon=False,
    columnspacing=1.4,
    handletextpad=0.5,
    labelspacing=0.6
)

fig.subplots_adjust(
    left=0.09,
    right=0.98,
    bottom=0.09,
    top=0.84,
    wspace=0.32,
    hspace=0.35
)

plt.show()